# Final Model Export and Website Integration

## Project
An Explainable Knowledge Graph-Driven Hybrid Recommender for Intelligent Task Allocation in Distributed Onboarding Operations

## Objective

The purpose of this notebook is to prepare the final recommendation system for integration with the web application.

The notebook will:

- Load the final recommender components
- Load the best hybrid weights selected during validation
- Create a reusable recommendation function for new onboarding tasks
- Generate Top-N agent recommendations and explanations
- Test the final model
- Export the required model artifacts for the Flask web application

The website will load the exported model locally and will not depend on a running Google Colab session.

## 1. Mounting Google Drive

Google Drive is mounted to access the trained recommendation components and evaluation results.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 2. Importing Required Libraries

The required Python libraries are imported for model loading, numerical calculations, recommendation generation and model export.

In [ ]:
import os
import json
import joblib
import shutil
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

## 3. Defining Deployment Paths

The model, processed data and deployment folders are defined.

The deployment folder will contain the files required by the Flask web application.

In [ ]:
BASE_PATH = (
    "/content/drive/MyDrive/"
    "task-allocation-recommender"
)

PROCESSED_PATH = os.path.join(
    BASE_PATH,
    "data",
    "onboarding",
    "processed"
)

MODEL_PATH = os.path.join(
    BASE_PATH,
    "models"
)

DEPLOYMENT_PATH = os.path.join(
    BASE_PATH,
    "deployment_artifacts"
)

os.makedirs(
    DEPLOYMENT_PATH,
    exist_ok=True
)

print("Models:", MODEL_PATH)
print("Deployment:", DEPLOYMENT_PATH)

Models: /content/drive/MyDrive/task-allocation-recommender/models
Deployment: /content/drive/MyDrive/task-allocation-recommender/deployment_artifacts


## 4. Loading the Final Recommendation Model

The recommendation components created in Notebook 02 and the best hybrid weights selected in Notebook 03 are loaded.

These components will be packaged into the final model used by the web application.

In [ ]:
components = joblib.load(
    os.path.join(
        MODEL_PATH,
        "recommender_components.joblib"
    )
)

In [ ]:
with open(
    os.path.join(
        MODEL_PATH,
        "best_hybrid_weights.json"
    ),
    "r"
) as file:

    BEST_WEIGHTS = json.load(file)

In [ ]:
tasks = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "onboarding_tasks.parquet"
    )
)

In [ ]:
tfidf_vectorizer = (
    components[
        "tfidf_vectorizer"
    ]
)

agent_skill_vectors = (
    components[
        "agent_skill_vectors"
    ]
)

agents = (
    components[
        "agents"
    ].copy()
)

collaborative_scores = (
    components[
        "collaborative_scores"
    ]
)

profile_to_column = (
    components[
        "profile_to_column"
    ]
)

historical_task_types = (
    components[
        "historical_task_types"
    ]
)

In [ ]:
print(
    "Agents:",
    len(agents)
)

print(
    "Best weights:",
    BEST_WEIGHTS
)

Agents: 500
Best weights: {'content': 0.25, 'collaborative': 0.2, 'knowledge_graph': 0.4, 'operational': 0.15}


## 5. Preparing Recommendation Helper Functions

Helper functions are created to process skills and calculate skill similarity between tasks and onboarding agents.

In [ ]:
def split_skills(skill_text):

    if pd.isna(skill_text):
        return []

    return [
        skill.strip()
        for skill
        in str(skill_text).split(",")
        if skill.strip()
    ]

In [ ]:
def calculate_skill_match(
    task_skills,
    agent_skills
):

    required = set(
        split_skills(
            task_skills
        )
    )

    available = set(
        split_skills(
            agent_skills
        )
    )

    if len(required) == 0:
        return 0

    return (
        len(
            required.intersection(
                available
            )
        )
        / len(required)
    )

## 6. Preparing Operational Agent Scores

The final model considers availability, workload and historical performance when ranking agents.

These operational factors are recreated so that the website can use the same scoring approach as the evaluated model.

In [ ]:
agents[
    "availability_score"
] = (
    agents[
        "availability_status"
    ]
    .astype(str)
    .str.lower()
    .eq("available")
    .astype(float)
)

In [ ]:
max_workload = (
    agents[
        "current_workload"
    ].max()
)

if max_workload == 0:
    max_workload = 1

agents[
    "workload_score"
] = (
    1
    -
    (
        agents[
            "current_workload"
        ]
        / max_workload
    )
)

In [ ]:
agents[
    "performance_score"
] = (
    agents[
        "performance_score"
    ]
    .fillna(
        agents[
            "performance_score"
        ].mean()
    )
    .clip(0, 1)
)

In [ ]:
agents[
    "operational_score"
] = (
    0.40
    * agents[
        "availability_score"
    ]

    + 0.35
    * agents[
        "workload_score"
    ]

    + 0.25
    * agents[
        "performance_score"
    ]
)

## 7. Creating Task Profiles for New Tasks

The deployed system must also support newly created onboarding tasks that were not part of the original dataset.

A task profile is therefore generated dynamically from the task type, onboarding channel and required skills.

In [ ]:
def create_task_profile(
    task_type,
    channel,
    required_skills
):

    return (
        str(task_type)
        + " | "
        + str(channel)
        + " | "
        + str(required_skills)
    )

## 8. Recommendation Explanation Function

The deployed recommender generates a human-readable explanation for each recommended agent.

The explanation uses factors such as matching skills, location, historical task experience, availability, workload and performance.

In [ ]:
def generate_explanation(
    task,
    agent
):

    reasons = []

    required = set(
        split_skills(
            task[
                "required_skills"
            ]
        )
    )

    available = set(
        split_skills(
            agent[
                "skills"
            ]
        )
    )

    matched = sorted(
        required.intersection(
            available
        )
    )

    if matched:

        reasons.append(
            "matches required skills: "
            + ", ".join(matched)
        )

    if (
        str(agent["county"])
        ==
        str(task["county"])
    ):

        reasons.append(
            "is located in the same county"
        )

    history = historical_task_types[
        (
            historical_task_types[
                "agent_id"
            ]
            == agent["agent_id"]
        )
        &
        (
            historical_task_types[
                "task_type"
            ]
            == task["task_type"]
        )
    ]

    if len(history) > 0:

        previous_tasks = int(
            history.iloc[0][
                "count"
            ]
        )

        reasons.append(
            f"has handled {previous_tasks} "
            "similar historical tasks"
        )

    if (
        str(
            agent[
                "availability_status"
            ]
        ).lower()
        == "available"
    ):

        reasons.append(
            "is currently available"
        )

    if (
        agent[
            "current_workload"
        ] <= 2
    ):

        reasons.append(
            "has a low current workload"
        )

    return "; ".join(reasons)

## 9. Final Recommendation Function

The final recommendation function accepts the details of a new onboarding task and returns the Top-N most suitable onboarding agents.

The function combines:

- Content-based similarity
- Collaborative filtering
- Knowledge graph relationships
- Operational factors

The output also contains a human-readable explanation for each recommendation.

In [ ]:
def recommend_agents(
    task,
    top_k=5
):

    task = task.copy()

    if "task_profile" not in task:

        task[
            "task_profile"
        ] = create_task_profile(
            task["task_type"],
            task["channel"],
            task["required_skills"]
        )

    # -------------------------
    # Content score
    # -------------------------

    task_vector = (
        tfidf_vectorizer
        .transform(
            [
                str(
                    task[
                        "required_skills"
                    ]
                )
            ]
        )
    )

    content_score = (
        cosine_similarity(
            task_vector,
            agent_skill_vectors
        )
        .flatten()
    )

    # -------------------------
    # Collaborative score
    # -------------------------

    task_profile = (
        task[
            "task_profile"
        ]
    )

    if (
        task_profile
        in profile_to_column
    ):

        column = (
            profile_to_column[
                task_profile
            ]
        )

        collaborative_score = (
            collaborative_scores[
                :,
                column
            ]
        )

    else:

        collaborative_score = (
            np.zeros(
                len(agents)
            )
        )

    # -------------------------
    # Knowledge Graph score
    # -------------------------

    skill_score = np.array(
        [
            calculate_skill_match(
                task[
                    "required_skills"
                ],
                skills
            )

            for skills
            in agents["skills"]
        ]
    )

    location_score = (
        agents["county"]
        .astype(str)
        .eq(
            str(
                task["county"]
            )
        )
        .astype(float)
        .to_numpy()
    )

    history = (
        historical_task_types[
            historical_task_types[
                "task_type"
            ]
            ==
            task[
                "task_type"
            ]
        ]
        .set_index(
            "agent_id"
        )["count"]
    )

    history_count = (
        agents["agent_id"]
        .map(history)
        .fillna(0)
        .to_numpy()
    )

    max_history = (
        history_count.max()
    )

    if max_history == 0:
        max_history = 1

    task_type_score = (
        history_count
        / max_history
    )

    kg_score = (
        0.60
        * skill_score

        + 0.20
        * location_score

        + 0.20
        * task_type_score
    )

    # -------------------------
    # Operational score
    # -------------------------

    operational_score = (
        agents[
            "operational_score"
        ]
        .to_numpy()
    )

    # -------------------------
    # Final hybrid score
    # -------------------------

    final_score = (
        BEST_WEIGHTS[
            "content"
        ]
        * content_score

        + BEST_WEIGHTS[
            "collaborative"
        ]
        * collaborative_score

        + BEST_WEIGHTS[
            "knowledge_graph"
        ]
        * kg_score

        + BEST_WEIGHTS[
            "operational"
        ]
        * operational_score
    )

    ranking = np.argsort(
        -final_score
    )[:top_k]

    results = (
        agents.iloc[
            ranking
        ][
            [
                "agent_id",
                "agent_name",
                "county",
                "skills",
                "availability_status",
                "current_workload",
                "performance_score"
            ]
        ]
        .copy()
    )

    results[
        "match_percentage"
    ] = (
        final_score[
            ranking
        ]
        * 100
    ).round(2)

    results[
        "explanation"
    ] = [
        generate_explanation(
            task,
            agents.iloc[index]
        )
        for index in ranking
    ]

    results.insert(
        0,
        "rank",
        range(
            1,
            len(results) + 1
        )
    )

    return (
        results
        .reset_index(drop=True)
    )

## 10. Testing the Final Recommender with a New Task

A new onboarding task is created manually to simulate a task submitted by a Supervisor through the future web application.

In [ ]:
new_task = {
    "task_type":
        "Enhanced KYC Onboarding",

    "channel":
        "agent",

    "county":
        "Nairobi",

    "required_skills":
        (
            "Customer Onboarding, "
            "KYC Verification, "
            "Enhanced KYC Review, "
            "Identity Verification"
        ),

    "priority":
        "High"
}

In [ ]:
final_test_recommendations = (
    recommend_agents(
        new_task,
        top_k=5
    )
)

final_test_recommendations

,rank,agent_id,agent_name,county,skills,availability_status,current_workload,performance_score,match_percentage,explanation
0,1,AGT-0433,Onboarding Agent 433,Nairobi,"Customer Onboarding, KYC Verification, Enhance...",Available,0,0.556,75.42,"matches required skills: Customer Onboarding, ..."
1,2,AGT-0368,Onboarding Agent 368,Nairobi,"Customer Onboarding, KYC Verification, Enhance...",Available,1,0.518,71.17,"matches required skills: Customer Onboarding, ..."
2,3,AGT-0136,Onboarding Agent 136,Kajiado,"Customer Onboarding, KYC Verification, Enhance...",Available,0,0.569,70.38,"matches required skills: Customer Onboarding, ..."
3,4,AGT-0448,Onboarding Agent 448,Nairobi,"Customer Onboarding, KYC Verification, Enhance...",Available,2,0.545,70.19,"matches required skills: Customer Onboarding, ..."
4,5,AGT-0010,Onboarding Agent 10,Machakos,"Customer Onboarding, KYC Verification, Enhance...",Available,0,0.555,69.52,"matches required skills: Customer Onboarding, ..."


## 11. Exporting the Final Recommendation Model

The final model components and selected hybrid weights are packaged into one deployment file.

This file can later be loaded by the Flask backend of the web application.

In [ ]:
final_model_bundle = {
    "components":
        components,

    "best_weights":
        BEST_WEIGHTS,

    "agents":
        agents,

    "model_version":
        "1.0"
}

In [ ]:
final_model_path = os.path.join(
    DEPLOYMENT_PATH,
    "final_recommender.joblib"
)

joblib.dump(
    final_model_bundle,
    final_model_path
)

print(
    "Final model saved:",
    final_model_path
)

Final model saved: /content/drive/MyDrive/task-allocation-recommender/deployment_artifacts/final_recommender.joblib


In [ ]:
deployment_files = [
    "best_hybrid_weights.json",
    "evaluation_summary.json",
    "model_comparison.csv",
    "hybrid_weight_tuning.csv"
]

In [ ]:
for filename in deployment_files:

    source = os.path.join(
        MODEL_PATH,
        filename
    )

    if os.path.exists(source):

        shutil.copy2(
            source,
            os.path.join(
                DEPLOYMENT_PATH,
                filename
            )
        )

## 12. Model Deployment Documentation

A model manifest is created to document the inputs, outputs, methodology and important limitations of the deployed recommender.

In [ ]:
model_manifest = {
    "project":
        (
            "Explainable Knowledge Graph-Driven "
            "Hybrid Recommender for Intelligent "
            "Task Allocation in Distributed "
            "Onboarding Operations"
        ),

    "version":
        "1.0",

    "inputs": [
        "task_type",
        "channel",
        "county",
        "required_skills",
        "priority"
    ],

    "output":
        "Top-N recommended onboarding agents",

    "components": {
        "content_based":
            BEST_WEIGHTS[
                "content"
            ],

        "collaborative":
            BEST_WEIGHTS[
                "collaborative"
            ],

        "knowledge_graph":
            BEST_WEIGHTS[
                "knowledge_graph"
            ],

        "operational":
            BEST_WEIGHTS[
                "operational"
            ]
    },

    "evaluation": {
        "Precision@5":
            0.0700,

        "Recall@5":
            0.3499,

        "MAP@5":
            0.2186,

        "Explanation_Coverage":
            1.0,

        "Hybrid_Workload_Std":
            80.72,

        "Knowledge_Graph_Workload_Std":
            106.06
    },

    "limitations":
        [
            "The original onboarding dataset is synthetic.",
            "The original source represents Nigerian digital onboarding data.",
            "Kenyan counties are synthetic prototype adaptations.",
            "Onboarding agent profiles are synthetic.",
            "Historical task-agent assignments are synthetic.",
            "Experimental results do not represent performance in a real organisation."
        ]
}

In [ ]:
manifest_path = os.path.join(
    DEPLOYMENT_PATH,
    "model_manifest.json"
)

with open(
    manifest_path,
    "w"
) as file:

    json.dump(
        model_manifest,
        file,
        indent=4
    )

print(
    "Manifest saved:",
    manifest_path
)

Manifest saved: /content/drive/MyDrive/task-allocation-recommender/deployment_artifacts/model_manifest.json


In [ ]:
print(
    "DEPLOYMENT FILES"
)

for filename in os.listdir(
    DEPLOYMENT_PATH
):

    print(
        "-",
        filename
    )

DEPLOYMENT FILES
- final_recommender.joblib
- best_hybrid_weights.json
- evaluation_summary.json
- model_comparison.csv
- hybrid_weight_tuning.csv
- model_manifest.json


# Notebook 04 Summary

The final recommendation model was successfully prepared for web application integration.

The completed deployment stage includes:

- Loading the trained recommendation components
- Loading the validation-selected hybrid weights
- Creating a reusable function for recommending agents for new onboarding tasks
- Generating explainable Top-N recommendations
- Testing the recommender using a new sample task
- Packaging the final recommender into a Joblib file
- Saving evaluation results and model documentation

The machine-learning development stage is now complete.

The next stage is the development of the Flask web application and integration of the exported recommender.

In [ ]:
print(
    "NOTEBOOK 04 COMPLETE"
)

print(
    "\nFinal model:",
    final_model_path
)

print(
    "\nHybrid weights:",
    BEST_WEIGHTS
)

print(
    "\nDeployment files:",
    len(
        os.listdir(
            DEPLOYMENT_PATH
        )
    )
)

NOTEBOOK 04 COMPLETE

Final model: /content/drive/MyDrive/task-allocation-recommender/deployment_artifacts/final_recommender.joblib

Hybrid weights: {'content': 0.25, 'collaborative': 0.2, 'knowledge_graph': 0.4, 'operational': 0.15}

Deployment files: 6
